# Lecture 13 — Production-Level Machine Learning Potentials

**PHYG004 / PHY5006, 2026 Spring · Sogang University**  
Prof. Young Woo Choi

---

## Learning goals
1. Understand the formalism of **NequIP**, **MACE**, and **MatterSim**.
2. Load and run three production-grade pretrained models — **MACE-MP-0**, **SevenNet-0**, **MatterSim-1M** — through a unified ASE `Calculator` interface.
3. Compare predictions on a benchmark system (equation of state of bulk Si).
4. Run a short MD simulation with a foundation potential.

## Prerequisites
- Lecture 11 — Graph Neural Networks for Molecules
- Lecture 12 — Equivariant Networks with `e3nn-jax`

> **Runtime tip:** in Colab, switch to **Runtime → Change runtime type → T4 GPU** before installing. Most cells run on CPU, but MD and MatterSim inference are noticeably faster on GPU.

<!-- lecture13-visual:start:title-pes -->
<div align="center">
  <img src="images/ai/01_pes_bridge.webp" width="780"/>
  <br><em>ML potentials bridge high-accuracy electronic-structure data and long-time atomistic simulation.</em>
</div>
<!-- lecture13-visual:end:title-pes -->



## 0. Setup

MACE currently requires `e3nn==0.4.4`, while recent MatterSim releases declare `e3nn>=0.5.0`. The install cell below uses `uv pip` and keeps the MACE-compatible `e3nn` while installing MatterSim without replacing it. Run the setup cell once in a fresh runtime.


In [ ]:
# ~3-5 min on a fresh Colab; longer if it pulls a CUDA-matched torch
import subprocess
import sys

try:
    import uv  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])

def uv_pip_install(*packages: str) -> None:
    subprocess.check_call([
        sys.executable, "-m", "uv", "pip", "install",
        "--python", sys.executable,
        *packages,
    ])

# MACE pins e3nn==0.4.4. Install this stack first so the compatible e3nn wins.
uv_pip_install(
    "mace-torch==0.3.16",
    "sevenn==0.10.4",
    "ase==3.28.0",
    "matplotlib==3.10.9",
)

# MatterSim runtime dependencies, excluding e3nn so MACE checkpoints still load.
uv_pip_install(
    "azure-identity", "azure-storage-blob", "deprecated",
    "atomate2", "emmet-core", "loguru", "mp-api",
    "pydantic>=2.9.2", "pymatgen", "seekpath", "phonopy", "phono3py",
    "torch-runstats", "torchaudio", "torchvision", "wandb",
)
uv_pip_install("--no-deps", "mattersim==1.2.3")

In [ ]:
import os
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

import numpy as np
import torch
import matplotlib.pyplot as plt
import time

from ase import Atoms, units
from ase.build import bulk, molecule

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch:  {torch.__version__}")
print(f"device: {device}")
if device == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

## 1. Recap — what we want from a potential

For $N$ atoms with positions $\{\mathbf{R}_i\}$ and species $\{Z_i\}$, an interatomic potential is a scalar function
$$
E\!\left(\{\mathbf{R}_i, Z_i\}\right) \in \mathbb{R}, \qquad \mathbf{F}_i = -\nabla_{\mathbf{R}_i} E .
$$

**Required symmetries**
- Translation invariance: $E(\{\mathbf{R}_i + \mathbf{t}\}) = E(\{\mathbf{R}_i\})$.
- $O(3)$ invariance of $E$; forces transform **equivariantly** under rotations.
- Permutation invariance over atoms of the same species.

**Locality** — decompose the energy into per-atom contributions within a cutoff sphere of radius $r_c$:
$$
E = \sum_i E_i\!\left(\mathcal{N}(i)\right), \qquad \mathcal{N}(i) = \{ j : |\mathbf{R}_j - \mathbf{R}_i| < r_c \}.
$$

Modern MLPs differ mainly in **how the local environment is encoded** while preserving these symmetries by construction.

<!-- lecture13-visual:start:local-environment -->
<div align="center">
  <img src="images/diagrams/local_environment.png" width="820"/>
  <br><em>Locality converts a many-atom PES into a size-extensive sum of atomic energy contributions.</em>
</div>
<!-- lecture13-visual:end:local-environment -->



## 2. NequIP — E(3)-equivariant message passing

Batzner *et al.*, *Nat. Commun.* **13**, 2453 (2022).

**Features are direct sums of $SO(3)$ irreps**
$$
h_i \;=\; \bigoplus_{l} h_i^{(l)} \;\in\; \bigoplus_l \mathbb{R}^{2l+1}.
$$
Scalars live in $l{=}0$, vectors in $l{=}1$, …

**Equivariant message** from $j$ to $i$:
$$
m_{ij} \;=\; \sum_{l_f,\, l_o} R_{l_f l_o}(r_{ij})\,\Big[\, Y^{(l_f)}(\hat{\mathbf{r}}_{ij}) \otimes_{l_o} h_j \,\Big]
$$
- $R$ : radial MLP on $r_{ij}$ (with smooth cutoff).
- $Y^{(l_f)}$ : real spherical harmonics — angular information.
- $\otimes_{l_o}$ : Clebsch–Gordan tensor product, projected onto output irrep $l_o$.

**Equivariant update**
$$
h_i \;\leftarrow\; \sigma\!\Big(W\,h_i + \sum_{j \in \mathcal{N}(i)} m_{ij}\Big),
$$
where $\sigma$ acts as a regular nonlinearity on the $l{=}0$ part and as a *gated* activation on $l>0$ parts so equivariance is preserved.

**Energy** comes from the invariant part:
$$
E_i \;=\; \mathrm{MLP}\!\left( h_i^{(0)} \right), \qquad E \;=\; \sum_i E_i.
$$

Because every operation is a CG tensor product of spherical harmonics with equivariant features, NequIP is **provably $E(3)$-equivariant by construction**. Empirically, this gives strong data efficiency — kcal/mol accuracy can be reached with $\sim 10^3$ DFT structures.

<!-- lecture13-visual:start:nequip -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/02_equivariant_message.webp" width="360"/><br><em>Visual intuition: geometric features rotate together with the atomic environment.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/diagrams/nequip_message.png" width="360"/><br><em>Mechanism: radial filters, spherical harmonics, and tensor products form equivariant messages.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:nequip -->



## 3. MACE — many-body equivariant messages

Batatia *et al.*, *NeurIPS* (2022); MACE-MP: Batatia *et al.*, arXiv:2401.00096 (2024).

**Step 1 — Atomic basis** (two-body, linear in the environment, just like NequIP)
$$
A_i^{(k,\,l)} \;=\; \sum_{j \in \mathcal{N}(i)} R_k(r_{ij})\, Y^{(l)}(\hat{\mathbf{r}}_{ij}) \otimes h_j .
$$

**Step 2 — Many-body features** via tensor products of $A$'s
$$
B_i^{(\nu)} \;=\; \sum_{k_1,\dots,k_\nu}
\Big( A_i^{(k_1)} \otimes A_i^{(k_2)} \otimes \cdots \otimes A_i^{(k_\nu)} \Big)_{\text{coupled to } l_o} .
$$
This is the equivariant generalisation of the **Atomic Cluster Expansion** (Drautz, 2019). With body order $\nu$ per layer and $L$ layers, the effective body order reaches $(\nu{+}1)^L$ — two layers with $\nu{=}3$ already capture up to 16-body interactions.

**Energy** is a sum of contributions from each layer’s invariant features:
$$
E \;=\; \sum_i \sum_{L'} W^{(L')} \cdot B_i^{(L',\, l=0)} .
$$

**Why MACE is fast.** High effective body order *per layer* means **fewer message-passing layers** are needed → less indirect communication → lower latency on large boxes. MACE-MP-0 is a 2-layer, $\nu{=}3$ model trained on the MPtrj subset of Materials Project (~$1.6 \times 10^6$ DFT structures, 89 elements).

<!-- lecture13-visual:start:mace -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/03_mace_many_body.webp" width="360"/><br><em>Visual intuition: one local environment contains multiple coupled many-body motifs.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/diagrams/mace_body_order.png" width="360"/><br><em>MACE packs higher body order into fewer message-passing layers.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:mace -->



## 4. SevenNet — distributed NequIP

Park *et al.* (Seoul National University). Code: [github.com/MDIL-SNU/SevenNet](https://github.com/MDIL-SNU/SevenNet).

- E(3)-equivariant message passing in the NequIP / Allegro family.
- **ZBL short-range repulsion** baked in, which makes MD stable under aggressive perturbations or high pressure.
- Distributed inference across multiple GPUs — practical for $> 10^5$-atom MD.
- **SevenNet-0** is the universal MPtrj-trained checkpoint — comparable scope to MACE-MP-0.
- Variants: **SevenNet-MF** (multi-fidelity training), **SevenNet-l3i5** (deeper), **SevenNet-D3** (with Grimme-D3 dispersion).

Think of SevenNet as a Korean, HPC-friendly cousin of NequIP.

<!-- lecture13-visual:start:sevennet -->
<div align="center">
  <img src="images/diagrams/sevennet_domain.png" width="760"/>
  <br><em>Distributed inference: domain decomposition plus boundary-neighbor communication.</em>
</div>
<!-- lecture13-visual:end:sevennet -->



## 5. MatterSim — broad-coverage foundation model

Yang *et al.*, *MatterSim: A Deep Learning Atomistic Model Across Elements, Temperatures and Pressures*, arXiv:2405.04967 (2024). Microsoft Research.

- **Architecture** — M3GNet-based (graph + bond-angle features) for the 1M model; EquiformerV2 variant for the 5M model.
- **Training set** — $\sim 1.7 \times 10^7$ structures generated by **active learning** across $(T, P)$ space:
  - Temperature: $0 - 5000$ K
  - Pressure: $0 - 1000$ GPa
  - Bulk, surfaces, liquids, amorphous, defects.
- **Checkpoints** — `MatterSim-v1.0.0-1M` (fast) and `MatterSim-v1.0.0-5M` (more accurate).
- Designed for **downstream property prediction** (phonons, free energies, EOS) with fine-tuning hooks.

Compared with MACE-MP / SevenNet, MatterSim trades some near-equilibrium accuracy for **dramatically broader thermodynamic coverage**.

<!-- lecture13-visual:start:mattersim -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/04_foundation_models.webp" width="360"/><br><em>Foundation potentials learn reusable atomistic representations across many materials classes.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/diagrams/mattersim_tp.png" width="360"/><br><em>MatterSim emphasizes broad active-learning coverage over temperature and pressure.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:mattersim -->



## 6. Foundation potential ecosystem (cheat sheet)

| Model | Architecture | Training set | Scope | License |
|-------|--------------|--------------|-------|---------|
| **CHGNet** | GNN + magnetic moments | MPtrj | inorganic | BSD-3 |
| **M3GNet** | GNN (bond + angle) | MPF.2021 | inorganic | BSD-3 |
| **MACE-MP-0** | Equivariant MPNN + ACE | MPtrj | inorganic, 89 elements | MIT |
| **MACE-OFF** | Equivariant MPNN + ACE | SPICE + extras | organic / drug-like | MIT |
| **SevenNet-0** | E(3) MPNN | MPtrj | inorganic | GPL-3 |
| **MatterSim-1M / 5M** | M3GNet / EqV2 | active-learned, 17M | broad $(T,P)$ | MIT |
| **Allegro** | Local equivariant NN | task-specific | trained per system | MIT |
| **ORB** | Graph transformer | MPtrj | inorganic | non-commercial |
| **EquiformerV2** | Transformer on irreps | OC20 / OC22 | catalysis | MIT |

Benchmarks: [Matbench Discovery](https://matbench-discovery.materialsproject.org/). Be careful — **leaderboard accuracy is not the same as usefulness for your science problem**.

<!-- lecture13-visual:start:ecosystem -->
<div align="center">
  <img src="images/diagrams/ecosystem_map.png" width="820"/>
  <br><em>Model selection is a scope-throughput-accuracy tradeoff, not a leaderboard-only decision.</em>
</div>
<!-- lecture13-visual:end:ecosystem -->



## 7. Hands-on A — load three calculators

All three models expose an ASE `Calculator`, so `atoms.get_potential_energy()` works uniformly once a calculator is attached.


In [ ]:
# --- MACE-MP-0 ---
from mace.calculators import mace_mp

mace_calc = mace_mp(
    model="medium",          # "small" / "medium" / "large"
    default_dtype="float32",
    device=device,
    dispersion=False,
)
print("MACE-MP-0 loaded.")

In [ ]:
# --- SevenNet-0 ---
from sevenn.calculator import SevenNetCalculator

sevennet_calc = SevenNetCalculator(model="7net-0", device=device)
print("SevenNet-0 loaded.")

In [ ]:
# --- MatterSim-1M ---
from mattersim.forcefield import MatterSimCalculator

mattersim_calc = MatterSimCalculator(
    load_path="MatterSim-v1.0.0-1M.pth",   # downloaded automatically on first call
    device=device,
)
print("MatterSim-1M loaded.")

## 8. Hands-on B — single-point on a perturbed Si supercell

We rattle a $2\times 2\times 2$ diamond-Si supercell (64 atoms) and compare **energies, forces, and timing** across the three models.

> Note: each model uses its own energy reference, so **absolute energies are not directly comparable**. Forces are.

<!-- lecture13-visual:start:single-point -->
<div align="center">
  <img src="images/diagrams/si_supercell.png" width="820"/>
  <br><em>The same perturbed Si supercell is passed through three ASE calculators.</em>
</div>
<!-- lecture13-visual:end:single-point -->



In [ ]:
rng = np.random.default_rng(42)

si = bulk("Si", "diamond", a=5.43, cubic=True).repeat((2, 2, 2))  # 64 atoms
si.rattle(stdev=0.05, seed=42)
print(f"System: {len(si)} Si atoms, cell lengths = {si.cell.lengths().round(2)} Å")

def evaluate(calc, name, atoms):
    a = atoms.copy()
    a.calc = calc
    t0 = time.perf_counter()
    E = a.get_potential_energy()
    F = a.get_forces()
    dt = time.perf_counter() - t0
    fmax = np.linalg.norm(F, axis=1).max()
    print(f"{name:>14s}:  E = {E:9.3f} eV   |F|max = {fmax:5.2f} eV/Å   ({dt*1000:6.1f} ms)")
    return E, F

E_mace,      F_mace      = evaluate(mace_calc,      "MACE-MP-0",    si)
E_sevenn,    F_sevenn    = evaluate(sevennet_calc,  "SevenNet-0",   si)
E_mattersim, F_mattersim = evaluate(mattersim_calc, "MatterSim-1M", si)

In [ ]:
def force_rmse(A, B):
    return float(np.sqrt(np.mean((A - B) ** 2)))

print(f"RMSE(MACE,     SevenNet)  = {force_rmse(F_mace,    F_sevenn):.4f} eV/Å")
print(f"RMSE(MACE,     MatterSim) = {force_rmse(F_mace,    F_mattersim):.4f} eV/Å")
print(f"RMSE(SevenNet, MatterSim) = {force_rmse(F_sevenn,  F_mattersim):.4f} eV/Å")

## 9. Hands-on C — equation of state of bulk Si

Sweep the cell volume and recompute energy at each scale. Subtract each model's own minimum so the three curves can be overlaid. A quadratic fit near the minimum gives the **bulk modulus** $B_0 = V \, d^2 E / dV^2$.

<!-- lecture13-visual:start:eos -->
<div align="center">
  <img src="images/diagrams/eos_concept.png" width="820"/>
  <br><em>The curvature of the energy-volume curve determines the bulk modulus.</em>
</div>
<!-- lecture13-visual:end:eos -->



In [ ]:
si0 = bulk("Si", "diamond", a=5.43, cubic=True)

scales = np.linspace(0.92, 1.10, 11)
volumes = []
energies = {"MACE-MP-0": [], "SevenNet-0": [], "MatterSim-1M": []}

for s in scales:
    a = si0.copy()
    a.set_cell(a.cell * s, scale_atoms=True)
    volumes.append(a.get_volume() / len(a))   # Å^3 / atom
    for name, calc in [
        ("MACE-MP-0",    mace_calc),
        ("SevenNet-0",   sevennet_calc),
        ("MatterSim-1M", mattersim_calc),
    ]:
        a.calc = calc
        energies[name].append(a.get_potential_energy() / len(a))   # eV / atom

volumes = np.array(volumes)
for name in energies:
    energies[name] = np.array(energies[name]) - np.min(energies[name])

In [ ]:
plt.figure(figsize=(6.5, 4.2))
for name, e in energies.items():
    plt.plot(volumes, e, "o-", label=name)
plt.xlabel("Volume per atom (Å³)")
plt.ylabel("E − E$_\\mathrm{min}$  (eV/atom)")
plt.title("Bulk Si — EOS from three pretrained MLPs")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# Parabolic fit gives V0, E0, and B0 = V * d^2E/dV^2.
# 1 eV/Å^3 = 160.21766208 GPa.

def fit_eos(V, E):
    a, b, c = np.polyfit(V, E, 2)
    V0 = -b / (2 * a)
    E0 = a * V0 ** 2 + b * V0 + c
    B0 = 2 * a * V0 * 160.21766208
    return V0, E0, B0

print(f"{'Model':<15s} {'V0 (Å³/at)':>12s} {'B0 (GPa)':>10s}")
print("-" * 40)
for name, e in energies.items():
    V0, E0, B0 = fit_eos(volumes, e)
    print(f"{name:<15s} {V0:>12.3f} {B0:>10.1f}")
print("-" * 40)
print(f"{'Experiment':<15s} {20.0:>12.3f} {99.0:>10.1f}")

**Discussion**
- SevenNet and MatterSim are close to the experimental diamond-Si bulk modulus (~99 GPa), while this quick MACE-MP-0 EOS underestimates the curvature.
- Differences reflect the DFT functional used to label the training set, the fitted reference geometry, the model's inductive bias, and the limited unrelaxed EOS sweep used here.
- For any property prediction study, **always benchmark on a known reference before trusting predictions on new systems**.


## 10. Hands-on D — short NVT MD with MACE-MP

Run 200 fs of Langevin dynamics on a Si supercell at 1500 K. We log the instantaneous temperature and the potential energy per atom.

<!-- lecture13-visual:start:md -->
<div align="center">
  <img src="images/diagrams/md_thermostat.png" width="820"/>
  <br><em>In NVT dynamics, the MLP supplies forces while Langevin dynamics controls temperature.</em>
</div>
<!-- lecture13-visual:end:md -->



In [ ]:
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units as u

si_md = bulk("Si", "diamond", a=5.43, cubic=True).repeat((2, 2, 2))
si_md.calc = mace_calc
MaxwellBoltzmannDistribution(si_md, temperature_K=1500)

dyn = Langevin(
    si_md,
    timestep=1.0 * u.fs,
    temperature_K=1500,
    friction=0.01,
)

T_log, E_log = [], []
def record():
    T_log.append(si_md.get_temperature())
    E_log.append(si_md.get_potential_energy() / len(si_md))

dyn.attach(record, interval=1)

t0 = time.perf_counter()
dyn.run(200)
print(f"MD wall-time: {time.perf_counter() - t0:5.1f} s for 200 fs ({len(si_md)} atoms)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(T_log)
axes[0].axhline(1500, color="r", linestyle="--", alpha=0.5, label="target 1500 K")
axes[0].set_xlabel("Step (fs)")
axes[0].set_ylabel("Temperature (K)")
axes[0].set_title("Thermostat tracking")
axes[0].legend()

axes[1].plot(E_log)
axes[1].set_xlabel("Step (fs)")
axes[1].set_ylabel("E$_\\mathrm{pot}$ / atom  (eV)")
axes[1].set_title("Potential energy")

plt.tight_layout()

## 11. When to use which?

- **MatterSim** — broad $(T, P)$ coverage out of the box: high-pressure phase diagrams, hot liquids, melting curves. Largest training set among open foundation MLPs.
- **MACE-MP-0** — strong general-purpose baseline for inorganic structures near equilibrium. Cheap inference. Active community.
- **MACE-OFF** — drug-like organic molecules; the natural choice for biomolecular MD.
- **SevenNet** — large-scale parallel MD ($> 10^5$ atoms). The built-in ZBL repulsion keeps it stable under aggressive perturbations or high pressure.
- **Allegro** — train per-system for *production accuracy* once a foundation model has bootstrapped the dataset.

**Recommended workflow for a new project**
1. Sanity-check several foundation MLPs against DFT on a handful of representative configurations.
2. If accuracy is sufficient, use the best one as-is.
3. Otherwise, **fine-tune** on a small DFT dataset; active learning helps select informative structures.
4. For extreme accuracy or unusual chemistry, train a dedicated NequIP/MACE/Allegro model from scratch.

<!-- lecture13-visual:start:workflow -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/05_active_learning_loop.webp" width="360"/><br><em>Active learning loop: simulate, flag uncertainty, label with DFT, retrain.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/06_production_md.webp" width="360"/><br><em>Production deployment: large boxes, domain decomposition, and GPU-parallel inference.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:workflow -->



## 12. References

1. Batzner *et al.*, *Nat. Commun.* **13**, 2453 (2022). **NequIP**.
2. Batatia *et al.*, *NeurIPS* (2022); arXiv:2206.07697. **MACE**.
3. Batatia *et al.*, arXiv:2401.00096 (2024). **MACE-MP**.
4. Park *et al.*, *J. Chem. Theory Comput.* (2024). **SevenNet**.
5. Yang *et al.*, arXiv:2405.04967 (2024). **MatterSim**.
6. Drautz, *Phys. Rev. B* **99**, 014104 (2019). **Atomic Cluster Expansion**.
7. Musaelian *et al.*, *Nat. Commun.* **14**, 579 (2023). **Allegro**.
8. Deng *et al.*, *Nat. Mach. Intell.* (2023). **CHGNet**.
9. Chen & Ong, *Nat. Comput. Sci.* **2**, 718 (2022). **M3GNet**.

---

*End of Lecture 13.*
